# Introduction to Vector Databases with ChromaDB

This notebook introduces the core concepts behind vector databases and demonstrates how to use ChromaDB to store, search, and retrieve semantic information using embeddings.

## What is a vector database?

A vector database stores data as high-dimensional vectors (embeddings) rather than only raw text. These embeddings capture the semantic meaning of the content, so similar items appear close together in vector space.

This makes vector databases useful for:
- semantic search
- recommendation systems
- retrieval-augmented generation (RAG)
- clustering and similarity analysis

In this notebook, we will create a small ChromaDB collection, add documents, and query for similar text.


In [ ]:
# Install the required libraries if they are not already available
# !pip install chromadb sentence-transformers

import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

print(f"Chroma version: {chromadb.__version__}")


## Creating a ChromaDB client

ChromaDB can run in-memory or persist data to disk. For a local demo, we will use a folder on disk so the data remains available between runs.


In [ ]:
import os
from pathlib import Path

persist_dir = Path("./chroma_db")
persist_dir.mkdir(exist_ok=True)

client = chromadb.Client(Settings(
    persist_directory=str(persist_dir),
    anonymized_telemetry=False,
))

# Use Chroma's default embedding function for the demo
embedding_fn = embedding_functions.DefaultEmbeddingFunction()

collection = client.get_or_create_collection(
    name="demo_collection",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn,
)

print("Collection created: ", collection.name)
print("Collection count: ", collection.count())


## Adding sample documents

Each document is stored with metadata and an embedding. We can then query the collection to find semantically similar items.


In [ ]:
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A machine learning model learns patterns from data.",
    "Vector databases help retrieve semantically similar information.",
    "Cats are independent and curious animals.",
    "A neural network can process text and images.",
]

ids = [f"doc_{i}" for i in range(len(documents))]
metadatas = [
    {"topic": "general", "source": "demo"}
    for _ in documents
]

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
)

print("Inserted documents: ", collection.count())


In [ ]:
query = "Find similar text about machine learning and retrieval systems."

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print("Query: ", query)
print("\nTop matches:\n")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"Document: {doc}")
    print(f"Metadata: {meta}")
    print(f"Distance: {distance:.4f}")
    print("-" * 50)


## Understanding the result

The query returns the closest matching documents based on vector similarity. Typically, documents with similar semantic meaning appear first, even if the exact words differ.

### Practical uses

- Search across documents by meaning
- Build RAG pipelines with LLMs
- Detect duplicates or near-duplicates
- Recommend related content

### Key takeaway

Vector databases do not store raw keyword matches alone. Instead, they store embeddings so similarity can be measured mathematically in a high-dimensional space.

This small example is a simple introduction to a much broader field of semantic search and retrieval systems.
